# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL ([FAIR\u005e2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

First, let's list all record sets by their `@id`, and for each, list the fields and columns with their `@id` where available.

In [ ]:
# Show all record sets and their fields/columns
print('Record Sets in this dataset:')
for rset in dataset.record_sets:
    print(f"\nRecordSet @id: {rset['@id']}")
    print(f"  name: {rset.get('name', 'n/a')}")
    # Fields
    if 'field' in rset:
        print(f"  Fields:")
        fields = rset['field']
        if isinstance(fields, dict):
            fields = [fields]
        for field in fields:
            print(f"   - @id: {field['@id']}   label: {field.get('name', field.get('@id'))}")
    # Columns
    if 'column' in rset:
        print(f"  Columns:")
        cols = rset['column']
        if isinstance(cols, dict):
            cols = [cols]
        for col in cols:
            print(f"   - @id: {col['@id']}   label: {col.get('name', col.get('@id'))}")

### List a Few Records from a Record Set

Below, we loop through a few records from a chosen record set, using its `@id`. Please refer to the overview above to choose a specific `@id`. For this example, we substitute with the record set `@id` found in the overview step.

In [ ]:
# Example: Print out the first 2 records for a specific record set
# Substitute with a real @id as found above; here we use a hypothetical one for demonstration
record_set_id = None
if len(dataset.record_sets) > 0:
    record_set_id = dataset.record_sets[0]['@id']

if record_set_id:
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i == 1:
            break
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load data from all available record sets into separate DataFrames for analysis. Use record set and field/column `@id`s found above.

In [ ]:
# Extract dataframes for all record sets
record_set_ids = [rset['@id'] for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for: {record_set_id} (columns: {list(df.columns)})")
    except Exception as e:
        print(f"Could not load DataFrame for {record_set_id}: {str(e)}")

Let's print columns and preview data for one of the main record sets. For demonstration, we'll use the first record set `@id` found earlier.

In [ ]:
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print('Columns:', dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No dataframes were loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by categorical attributes. For illustration, we will use the first loaded dataframe, and select its numeric field by matching columns with numeric types.

In [ ]:
# EDA: Filtering, Normalization, and Grouping
import numpy as np

if len(dataframes) > 0:
    df = list(dataframes.values())[0]
    # Detect a numeric field
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Use a likely group field ('description', 'sample', 'accession', etc.)
        candidate_group_fields = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() < 20]
        group_field = candidate_group_fields[0] if candidate_group_fields else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')
    else:
        print('No numeric field found in the DataFrame for EDA.')
else:
    print('No dataframes loaded for EDA.')

## 5. Visualization
Visualize the distribution of the numeric field (if available) and group means (if grouping succeeded).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution
if len(dataframes) > 0:
    df = list(dataframes.values())[0]
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) > 0:
        numeric_field = numeric_cols[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
    else:
        print('No numeric field to plot.')
else:
    print('No DataFrames loaded for visualization.')

## 6. Conclusion
In this notebook, we loaded a FAIR\u005e2 Croissant dataset about mass spectrometry analysis of extracellular vesicles from stimulated human mast cells, listed its record sets and fields via their `@id`, loaded the data into DataFrames, performed basic filtering and normalization on numeric fields, grouped data when possible, and visualized field distributions. For further analysis, continue exploring the available fields and relationships using their `@id` as shown in this workflow.